<a href="https://colab.research.google.com/github/Marfall/PyTorchApproximation-Otus-5/blob/main/PyTorchApproximation_Otus_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание №5: Практика по PyTorch

**Цель:** получить навыки практической работы с PyTorch.

**Задача:** аппроксимировать функцию `sin(x + 2y) · exp(-(2x + y)²)` на диапазоне `[-10; 10]` с помощью регрессионной нейронной сети.

**План:**
1. Сгенерировать 20 000 случайных точек.
2. Разделить на train / val / test в пропорции 70% / 15% / 15%.
3. Обучить две модели с одинаковой архитектурой, но разными активациями: ReLU и Tanh.
4. Сравнить MSE на тестовой выборке.
5. Визуализировать истинную и предсказанные поверхности.

In [ ]:
# 1: Импорт библиотек и настройка окружения
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

## 2. Генерация данных

Создаются 20 000 точек со случайными координатами x и y в диапазоне [-10, 10]. Для каждой точки вычисляется значение целевой функции.

In [ ]:
# 3: Генерация данных
np.random.seed(42)
n_points = 20000

x = np.random.uniform(-10, 10, n_points)
y = np.random.uniform(-10, 10, n_points)

def target_function(x, y):
    """Целевая функция: sin(x + 2y) * exp(-(2x + y)^2)"""
    return np.sin(x + 2*y) * np.exp(-(2*x + y)**2)

z = target_function(x, y)

data = pd.DataFrame({'x': x, 'y': y, 'z': z})

print(f"Размер датасета: {data.shape}")
print(f"Диапазон x: [{x.min():.2f}, {x.max():.2f}]")
print(f"Диапазон y: [{y.min():.2f}, {y.max():.2f}]")
print(f"Диапазон z: [{z.min():.4f}, {z.max():.4f}]")
print(f"Среднее z: {z.mean():.4f}, стандартное отклонение z: {z.std():.4f}")

data.head()

## 4. Разведочный анализ (EDA)

Строятся гистограммы распределения x, y и z, а также 3D-визуализация истинной функции на равномерной сетке. Под графиками выводится сводная таблица с числовыми характеристиками функции.

In [ ]:
# 5: EDA — гистограммы, 3D-визуализация, числовые сводки
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(x, bins=50, ax=axes[0], color='steelblue', kde=True)
axes[0].set_title('Распределение x')
sns.histplot(y, bins=50, ax=axes[1], color='steelblue', kde=True)
axes[1].set_title('Распределение y')
sns.histplot(z, bins=50, ax=axes[2], color='steelblue', kde=True)
axes[2].set_title('Распределение z (целевая)')
plt.tight_layout()
plt.show()
plt.close('all')

x_grid = np.linspace(-10, 10, 100)
y_grid = np.linspace(-10, 10, 100)
X_grid, Y_grid = np.meshgrid(x_grid, y_grid)
Z_true = target_function(X_grid, Y_grid)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X_grid, Y_grid, Z_true, cmap='viridis', alpha=0.9, edgecolor='none')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Истинная функция: sin(x + 2y) · exp(-(2x + y)²)')
fig.colorbar(surf, shrink=0.5, aspect=10, label='z')
plt.show()
plt.close('all')

summary_eda = pd.DataFrame({
    'Показатель': [
        'Количество точек', 'Диапазон x', 'Диапазон y',
        'Среднее z', 'Стандартное отклонение z',
        'Минимум z', 'Максимум z', 'Абс. максимум |z|',
        'Перцентиль 0.5%', 'Перцентиль 99.5%', 'Доля |z| < 0.01'
    ],
    'Значение': [
        n_points,
        f"[{x.min():.2f}, {x.max():.2f}]",
        f"[{y.min():.2f}, {y.max():.2f}]",
        f"{z.mean():.6f}",
        f"{z.std():.6f}",
        f"{z.min():.6f}",
        f"{z.max():.6f}",
        f"{np.abs(z).max():.6f}",
        f"{np.percentile(z, 0.5):.6f}",
        f"{np.percentile(z, 99.5):.6f}",
        f"{(np.abs(z) < 0.01).mean()*100:.2f}%"
    ]
})

print("=== ЧИСЛОВАЯ СВОДКА ПО ДАННЫМ ===")
print(summary_eda.to_string(index=False))

## 6. Промежуточные выводы по EDA

**Что мы видим на графиках:**

- **Распределение x и y:** равномерное на [-10, 10], что ожидаемо при случайной генерации.
- **Распределение z:** сильно сконцентрировано около нуля — большинство значений близки к 0, экстремумы редки. Это следствие быстрого затухания экспоненциального множителя.
- **3D-поверхность истинной функции:** ярко выраженный «гребень» вдоль линии, где 2x + y ≈ 0; вне этой линии функция практически обнуляется.

**Числовое подтверждение:**
- Диапазон x: [X_MIN, X_MAX]
- Диапазон y: [Y_MIN, Y_MAX]
- Среднее z: Z_MEAN
- Стандартное отклонение z: Z_STD
- Абсолютный максимум |z|: Z_ABSMAX
- Доля |z| < 0.01: PERCENT_SMALL

**Вывод:** функция имеет локализованный характер. Модель должна научиться фокусироваться на узкой области входа, где значения ненулевые.

## 7. Разделение данных и нормализация

Датасет делится на train (70%), val (15%) и test (15%). Входные признаки и целевая переменная стандартизируются через `StandardScaler`. Создаются `TensorDataset` и `DataLoader`.

In [ ]:
# 8: Разделение данных, нормализация, DataLoader
X_all = data[['x', 'y']].values
y_all = data['z'].values.reshape(-1, 1)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f"Train: {X_train.shape[0]} ({X_train.shape[0]/len(X_all)*100:.1f}%)")
print(f"Val:   {X_val.shape[0]} ({X_val.shape[0]/len(X_all)*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} ({X_test.shape[0]/len(X_all)*100:.1f}%)")

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_val_scaled = scaler_y.transform(y_val)
y_test_scaled = scaler_y.transform(y_test)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_t = torch.tensor(y_val_scaled, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test_scaled, dtype=torch.float32)

batch_size = 256
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)

print(f"Размер батча: {batch_size}")
print(f"Количество батчей в train: {len(train_loader)}")

## 9. Архитектура модели

Используется полносвязная нейронная сеть (MLP) фиксированной архитектуры: 2 → 128 → 128 → 1. Меняется только функция активации: ReLU или Tanh.

In [ ]:
# 10: Определение класса MLP
class MLP(nn.Module):
    """Многослойный перцептрон для регрессии."""
    def __init__(self, activation='relu'):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(2, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 1)

        if activation == 'relu':
            self.act = nn.ReLU()
        elif activation == 'tanh':
            self.act = nn.Tanh()
        else:
            raise ValueError(f"Неизвестная активация: {activation}")

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.fc3(x)
        return x

## 10. Функция обучения

Реализован цикл обучения с валидацией и ранней остановкой. Оптимизатор — Adam, функция потерь — MSE. Лучшая модель по val_loss сохраняется.

In [ ]:
# 12: Функция обучения с ранней остановкой
def train_model(model, train_loader, val_loader, epochs=200, lr=0.001, patience=20):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                val_loss += loss.item() * X_batch.size(0)
        val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_model_state = model.state_dict().copy()
        else:
            epochs_no_improve += 1

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

        if epochs_no_improve >= patience:
            print(f"\nРанняя остановка на эпохе {epoch+1}. Лучшая val loss: {best_val_loss:.6f}")
            break

    model.load_state_dict(best_model_state)
    return model, train_losses, val_losses

## 13. Обучение модели с активацией ReLU

In [ ]:
# 14: Обучение модели с ReLU
print("=" * 60)
print("Обучение модели с активацией ReLU")
print("=" * 60)

model_relu = MLP(activation='relu').to(device)
model_relu, train_losses_relu, val_losses_relu = train_model(model_relu, train_loader, val_loader)

## 15. Обучение модели с активацией Tanh

In [ ]:
# 16: Обучение модели с Tanh
print("=" * 60)
print("Обучение модели с активацией Tanh")
print("=" * 60)

model_tanh = MLP(activation='tanh').to(device)
model_tanh, train_losses_tanh, val_losses_tanh = train_model(model_tanh, train_loader, val_loader)

## 17. Оценка на тестовой выборке

Вычисляется MSE в исходном масштабе целевой переменной для обеих моделей. Определяется победитель.

In [ ]:
# 18: Оценка моделей на тестовой выборке
def evaluate_model(model, test_loader, scaler_y):
    model.eval()
    criterion = nn.MSELoss()
    test_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            test_loss += loss.item() * X_batch.size(0)
            all_preds.append(y_pred.cpu().numpy())
            all_targets.append(y_batch.cpu().numpy())

    test_loss /= len(test_loader.dataset)

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    preds_orig = scaler_y.inverse_transform(all_preds)
    targets_orig = scaler_y.inverse_transform(all_targets)
    mse_orig = np.mean((preds_orig - targets_orig) ** 2)

    return test_loss, mse_orig

test_loss_relu, mse_relu = evaluate_model(model_relu, test_loader, scaler_y)
test_loss_tanh, mse_tanh = evaluate_model(model_tanh, test_loader, scaler_y)

print("\n" + "=" * 60)
print("РЕЗУЛЬТАТЫ НА ТЕСТОВОЙ ВЫБОРКЕ")
print("=" * 60)
print(f"ReLU — MSE (исходный масштаб): {mse_relu:.6f}")
print(f"Tanh — MSE (исходный масштаб): {mse_tanh:.6f}")

if mse_relu < mse_tanh:
    winner = "ReLU"
    diff = mse_tanh - mse_relu
    print(f"\nПобедитель: ReLU (MSE на {diff:.6f} меньше)")
else:
    winner = "Tanh"
    diff = mse_relu - mse_tanh
    print(f"\nПобедитель: Tanh (MSE на {diff:.6f} меньше)")

## 19. Кривые обучения

Сравнение динамики train и val loss для обеих моделей. Под графиком — сводная таблица с финальными и минимальными значениями.

In [ ]:
# 20: Кривые обучения + числовая сводка
plt.figure(figsize=(10, 5))
sns.lineplot(x=range(len(train_losses_relu)), y=train_losses_relu, label='ReLU — Train', linewidth=2)
sns.lineplot(x=range(len(val_losses_relu)), y=val_losses_relu, label='ReLU — Val', linewidth=2, linestyle='--')
sns.lineplot(x=range(len(train_losses_tanh)), y=train_losses_tanh, label='Tanh — Train', linewidth=2)
sns.lineplot(x=range(len(val_losses_tanh)), y=val_losses_tanh, label='Tanh — Val', linewidth=2, linestyle='--')
plt.xlabel('Эпоха')
plt.ylabel('MSE Loss (нормированный)')
plt.title('Кривые обучения')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
plt.close('all')

summary_loss = pd.DataFrame({
    'Модель': ['ReLU', 'ReLU', 'Tanh', 'Tanh'],
    'Метрика': ['Финальный Train Loss', 'Минимальный Val Loss',
                'Финальный Train Loss', 'Минимальный Val Loss'],
    'Значение': [
        f"{train_losses_relu[-1]:.6f}",
        f"{min(val_losses_relu):.6f}",
        f"{train_losses_tanh[-1]:.6f}",
        f"{min(val_losses_tanh):.6f}"
    ],
    'Эпох обучено': [
        len(train_losses_relu), len(val_losses_relu),
        len(train_losses_tanh), len(val_losses_tanh)
    ]
})

print("=== ЧИСЛОВАЯ СВОДКА ПО ОБУЧЕНИЮ ===")
print(summary_loss.to_string(index=False))

## 21. Промежуточные выводы по кривым обучения

**Что мы видим на графиках:**

- **ReLU:** кривые train и val loss снижаются, но могут присутствовать небольшие колебания.
- **Tanh:** обычно кривые более гладкие, сходимость стабильная.
- Разрыв между train и val показывает, есть ли переобучение.

**Числовое подтверждение:**
- Финальный train loss (ReLU): RELU_TRAIN
- Минимальный val loss (ReLU): RELU_VAL
- Финальный train loss (Tanh): TANH_TRAIN
- Минимальный val loss (Tanh): TANH_VAL
- Количество эпох до ранней остановки (ReLU): RELU_EPOCHS
- Количество эпох до ранней остановки (Tanh): TANH_EPOCHS

**Вывод:** если разрыв между train и val невелик — модель не переобучена. Если val loss перестала падать раньше train — сработала ранняя остановка.

## 22. Визуализация результатов

Построены три 3D-поверхности: истинная функция, предсказание ReLU и предсказание Tanh. Под графиком — числовое сравнение поверхностей.

In [ ]:
# 23: 3D-визуализация + числовое сравнение поверхностей
x_grid = np.linspace(-10, 10, 100)
y_grid = np.linspace(-10, 10, 100)
X_grid, Y_grid = np.meshgrid(x_grid, y_grid)
Z_true = target_function(X_grid, Y_grid)

grid_points = np.column_stack([X_grid.ravel(), Y_grid.ravel()])
grid_points_scaled = scaler_X.transform(grid_points)
grid_points_t = torch.tensor(grid_points_scaled, dtype=torch.float32).to(device)

model_relu.eval()
model_tanh.eval()
with torch.no_grad():
    pred_relu = model_relu(grid_points_t).cpu().numpy()
    pred_tanh = model_tanh(grid_points_t).cpu().numpy()

pred_relu = scaler_y.inverse_transform(pred_relu).reshape(X_grid.shape)
pred_tanh = scaler_y.inverse_transform(pred_tanh).reshape(X_grid.shape)

fig = plt.figure(figsize=(18, 6))

ax1 = fig.add_subplot(131, projection='3d')
surf1 = ax1.plot_surface(X_grid, Y_grid, Z_true, cmap='viridis', alpha=0.9, edgecolor='none')
ax1.set_title('Истинная функция', pad=10)
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')
fig.colorbar(surf1, ax=ax1, shrink=0.5, aspect=10, label='z')

ax2 = fig.add_subplot(132, projection='3d')
surf2 = ax2.plot_surface(X_grid, Y_grid, pred_relu, cmap='plasma', alpha=0.9, edgecolor='none')
ax2.set_title(f'Предсказание ReLU\nMSE = {mse_relu:.6f}', pad=10)
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('z')
fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=10, label='z')

ax3 = fig.add_subplot(133, projection='3d')
surf3 = ax3.plot_surface(X_grid, Y_grid, pred_tanh, cmap='plasma', alpha=0.9, edgecolor='none')
ax3.set_title(f'Предсказание Tanh\nMSE = {mse_tanh:.6f}', pad=10)
ax3.set_xlabel('x'); ax3.set_ylabel('y'); ax3.set_zlabel('z')
fig.colorbar(surf3, ax=ax3, shrink=0.5, aspect=10, label='z')

plt.tight_layout()
plt.show()
plt.close('all')

mse_grid_relu = np.mean((pred_relu - Z_true) ** 2)
mse_grid_tanh = np.mean((pred_tanh - Z_true) ** 2)

summary_surfaces = pd.DataFrame({
    'Метрика': [
        'Среднее значение', 'Стандартное отклонение',
        'Минимум', 'Максимум',
        'MSE относительно истинной', 'Макс. абсолютная ошибка'
    ],
    'Истинная': [
        f"{Z_true.mean():.6f}", f"{Z_true.std():.6f}",
        f"{Z_true.min():.6f}", f"{Z_true.max():.6f}",
        '—', '—'
    ],
    'ReLU': [
        f"{pred_relu.mean():.6f}", f"{pred_relu.std():.6f}",
        f"{pred_relu.min():.6f}", f"{pred_relu.max():.6f}",
        f"{mse_grid_relu:.6f}", f"{np.abs(pred_relu - Z_true).max():.6f}"
    ],
    'Tanh': [
        f"{pred_tanh.mean():.6f}", f"{pred_tanh.std():.6f}",
        f"{pred_tanh.min():.6f}", f"{pred_tanh.max():.6f}",
        f"{mse_grid_tanh:.6f}", f"{np.abs(pred_tanh - Z_true).max():.6f}"
    ]
})

print("=== ЧИСЛОВАЯ СВОДКА ПО ПОВЕРХНОСТЯМ ===")
print(summary_surfaces.to_string(index=False))

## 24. Промежуточные выводы по 3D-поверхностям

**Что мы видим на графиках:**

- **Истинная функция:** узкий «хребет» вдоль линии 2x + y ≈ 0, резкое затухание во все стороны.
- **Предсказание ReLU:** поверхность в целом повторяет форму, но могут быть видны «изломы» и менее гладкие переходы.
- **Предсказание Tanh:** поверхность более гладкая, ближе к истинной, особенно в областях резкого затухания.

**Числовое подтверждение:**
- MSE на тестовой выборке (ReLU): MSE_RELU
- MSE на тестовой выборке (Tanh): MSE_TANH
- MSE относительно истинной поверхности (ReLU, на сетке): MSE_GRID_RELU
- MSE относительно истинной поверхности (Tanh, на сетке): MSE_GRID_TANH
- Максимальная абсолютная ошибка (ReLU): MAXERR_RELU
- Максимальная абсолютная ошибка (Tanh): MAXERR_TANH

**Вывод:** обе модели справились с задачей, но разница в MSE и визуальной гладкости позволяет выбрать лучшую. Победитель по MSE — [ВСТАВИТЬ ПОСЛЕ РЕЗУЛЬТАТОВ].

## 25. Итоговые выводы

Обобщаются результаты обеих моделей. Ниже выводится сводная таблица со всеми ключевыми метриками, которые затем используются для формулировки итоговых выводов.

In [ ]:
# 26: Итоговая сводка всех ключевых чисел
final_summary = pd.DataFrame({
    'Параметр': [
        'Количество точек в датасете',
        'Train / Val / Test',
        'Архитектура',
        'Активации',
        'Оптимизатор',
        'Learning rate',
        'Batch size',
        'Эпох обучено (ReLU)',
        'Эпох обучено (Tanh)',
        'Финальный train loss (ReLU)',
        'Минимальный val loss (ReLU)',
        'Финальный train loss (Tanh)',
        'Минимальный val loss (Tanh)',
        'MSE на тесте (ReLU)',
        'MSE на тесте (Tanh)',
        'MSE на сетке (ReLU)',
        'MSE на сетке (Tanh)',
        'Макс. ошибка (ReLU)',
        'Макс. ошибка (Tanh)',
        'Победитель по MSE'
    ],
    'Значение': [
        n_points,
        f"{X_train.shape[0]} / {X_val.shape[0]} / {X_test.shape[0]}",
        "2 → 128 → 128 → 1 (MLP)",
        "ReLU и Tanh",
        "Adam",
        "0.001",
        batch_size,
        len(train_losses_relu),
        len(train_losses_tanh),
        f"{train_losses_relu[-1]:.6f}",
        f"{min(val_losses_relu):.6f}",
        f"{train_losses_tanh[-1]:.6f}",
        f"{min(val_losses_tanh):.6f}",
        f"{mse_relu:.6f}",
        f"{mse_tanh:.6f}",
        f"{mse_grid_relu:.6f}",
        f"{mse_grid_tanh:.6f}",
        f"{np.abs(pred_relu - Z_true).max():.6f}",
        f"{np.abs(pred_tanh - Z_true).max():.6f}",
        winner
    ]
})

print("=" * 60)
print("ИТОГОВАЯ СВОДКА ПО ВСЕМ КЛЮЧЕВЫМ ЧИСЛАМ")
print("=" * 60)
print(final_summary.to_string(index=False))